# 07 — Pruebas y resultados del Laboratorio 4

Ejecuta los **20 escenarios** reproducibles del notebook completo: 14 por texto y 6 por audio. Las seis pruebas de voz cubren los casos de la rúbrica; las interacciones incluyen varios turnos. Se conserva registro de transcripción, correcciones, herramientas, estado final y latencias.

Antes de ejecutar, prepara `data/scenario_audio/<VOICE_ID>/E01_t1.wav` … `E06_tN.wav`. Los nombres exactos están en `ESCENARIOS`.

Ejecuta una sola vez la evaluación final después de fijar los parámetros en desarrollo.

In [ ]:
%pip -q install -r requirements-lab4-modular.txt

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "lab4_asistente").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src" / "lab4_asistente").is_dir()), ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Raíz del proyecto:", ROOT)
from pathlib import Path
import importlib.metadata
import pandas as pd
from IPython.display import display
from lab4_asistente import config as C
from lab4_asistente.escenarios import ESCENARIOS, ejecutar
from lab4_asistente.evaluacion import (tabla_escenarios, resumen_escenarios,
    tabla_asr, tabla_latencias, tabla_calidad_tts)
from lab4_asistente.runtime import cargar_sistema, cargar_modelos_voz
from lab4_asistente.voz import Semejanza


## Cargar modelos y verificar que estén los audios requeridos

In [ ]:
COMP = cargar_sistema(cargar_llm=True)
SIS = COMP["sistema"]
VOICE_ID = "winz"
ASR, TTS, cargas_voz = cargar_modelos_voz(VOICE_ID, COMP["dispositivo"])
CARGAS_INICIALES = {**COMP["cargas"], **cargas_voz}
RUTA_AUDIOS = C.RUTA_AUDIO_ESCENARIOS / VOICE_ID
AUDIO_REQUERIDO = [(e["id"], t["audio"]) for e in ESCENARIOS if e["modo"] == "voz" for t in e["turnos"]]
FALTAN = [nombre for _, nombre in AUDIO_REQUERIDO if not (RUTA_AUDIOS / nombre).exists()]
print("Cargas iniciales:", CARGAS_INICIALES)
print("Audios requeridos:", len(AUDIO_REQUERIDO), "| faltantes:", len(FALTAN))
if FALTAN:
    display(pd.DataFrame({"audios_faltantes": FALTAN}))


## Ejecutar los 14 escenarios de texto

In [ ]:
C.RUTA_ESCENARIOS.mkdir(parents=True, exist_ok=True)
RES_TEXTO = []
for e in [x for x in ESCENARIOS if x["modo"] == "texto"]:
    RES_TEXTO.append(ejecutar(e, SIS, "texto", C.RUTA_ESCENARIOS / "texto" / e["id"]))
print("Escenarios de texto ejecutados:", len(RES_TEXTO))


## Ejecutar los seis escenarios con audio

Las grabaciones se conservan junto con transcripciones y operaciones. Se ejecutan también con sus textos correctos para separar errores del ASR de errores del agente.

In [ ]:
assert not FALTAN, "Completa las grabaciones requeridas antes de ejecutar la evaluación de voz."
RES_VOZ, RES_VOZ_TEXTO = [], []
for e in [x for x in ESCENARIOS if x["modo"] == "voz"]:
    carpeta = C.RUTA_ESCENARIOS / "voz_audio" / e["id"]
    RES_VOZ.append(ejecutar(e, SIS, "voz", carpeta, RUTA_AUDIOS, ASR, TTS, sintetizar=True))
    carpeta_txt = C.RUTA_ESCENARIOS / "voz_transcripcion_correcta" / e["id"]
    RES_VOZ_TEXTO.append(ejecutar(e, SIS, "texto", carpeta_txt))
print("Escenarios de voz:", len(RES_VOZ), "| replay con transcripción correcta:", len(RES_VOZ_TEXTO))


## Resumen, incumplimientos y métricas de ASR

In [ ]:
TODOS = RES_TEXTO + RES_VOZ
MATRIZ = tabla_escenarios(TODOS)
display(MATRIZ)
print("Resumen:", resumen_escenarios(TODOS))
ASR_METRICAS = tabla_asr(RES_VOZ)
display(ASR_METRICAS)
if len(ASR_METRICAS):
    print("WER medio:", ASR_METRICAS.WER.mean())
    print("CER medio:", ASR_METRICAS.CER.mean())
    print("Errores de cantidades/importes/productos:",
          ASR_METRICAS.err_cantidades.sum(), ASR_METRICAS.err_importes.sum(), ASR_METRICAS.err_productos.sum())


## Latencia por etapa y evaluación de voz sintetizada

In [ ]:
LATENCIAS = tabla_latencias(RES_VOZ)
display(LATENCIAS.groupby("etapa").segundos.agg(["mean", "median", "max"]).sort_values("mean", ascending=False))
print("Carga inicial separada:", CARGAS_INICIALES)
SEMEJANZA = Semejanza(C.MODELO_SEMEJANZA, "cpu")
REF_VOZ = C.RUTA_VOICE_SAMPLES / f"{VOICE_ID}.wav"
TTS_METRICAS = tabla_calidad_tts(RES_VOZ, C.RUTA_ESCENARIOS / "voz_audio", ASR, SEMEJANZA, REF_VOZ)
display(TTS_METRICAS)


## Análisis de cinco aciertos y cinco fallos

In [ ]:
ACIERTO = MATRIZ[MATRIZ.completado == True].head(5)
FALLO = MATRIZ[MATRIZ.completado == False].head(5)
print("Aciertos — describir qué condición verificó cada uno:")
display(ACIERTO)
print("Fallos — identificar la causa con el registro de operaciones:")
display(FALLO)


## Guardar resultados y configuración ejecutada

In [ ]:
C.RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
MATRIZ.to_csv(C.RUTA_RESULTADOS / "metricas_escenarios.csv", index=False)
ASR_METRICAS.to_csv(C.RUTA_RESULTADOS / "metricas_asr.csv", index=False)
LATENCIAS.to_csv(C.RUTA_RESULTADOS / "latencias_por_turno.csv", index=False)
TTS_METRICAS.to_csv(C.RUTA_RESULTADOS / "metricas_tts_por_respuesta.csv", index=False)
paquetes = ["torch", "torchaudio", "transformers", "sentence-transformers", "TTS", "pandas", "pyarrow", "rank-bm25"]
versiones = {}
for nombre in paquetes:
    try:
        versiones[nombre] = importlib.metadata.version(nombre)
    except importlib.metadata.PackageNotFoundError:
        versiones[nombre] = "no instalado"
(C.RUTA_RESULTADOS / "versiones.json").write_text(pd.Series(versiones).to_json(indent=2), encoding="utf-8")
print("Resultados guardados en:", C.RUTA_RESULTADOS)
